In [13]:
import pandas as pd
import sqlite3

In [14]:
base_connect = sqlite3.connect("../data/checking-logs.sqlite")

## Одним запросом создай в базе новую таблицу

In [15]:
query_drop = "DROP TABLE IF EXISTS datamart;"
base_connect.execute(query_drop)

In [16]:
datamart = pd.io.sql.read_sql(
    """
    SELECT 
        CASE WHEN c.uid LIKE 'user_%' THEN c.uid ELSE NULL END AS uid,
        CASE WHEN c.uid LIKE 'user_%' THEN c.labname ELSE NULL END AS labname,
        MIN(CASE WHEN c.uid LIKE 'user_%' THEN c.timestamp ELSE NULL END) AS first_commit_ts,
        p.first_view_ts
    FROM checker AS c
    LEFT JOIN (
        SELECT uid, MIN(datetime) AS first_view_ts 
        FROM pageviews 
        WHERE uid LIKE 'user_%'
        GROUP BY uid
    ) AS p ON c.uid = p.uid
    WHERE c.status = 'ready'
      AND c.numTrials = 1
      AND TRIM(c.labname) IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    GROUP BY c.uid, c.labname;
    """,
    base_connect,
    parse_dates=["first_commit_ts", "first_view_ts"],
)

datamart["first_commit_ts"] = datamart["first_commit_ts"].astype("datetime64[ns]")
datamart["first_view_ts"] = datamart["first_view_ts"].astype("datetime64[ns]")

datamart.to_sql("datamart", base_connect, if_exists="replace", index=False)

datamart.info()

<class 'pandas.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    str           
 1   labname          140 non-null    str           
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 4.7 KB


## С помощью методов Pandas создай два DataFrame: test и control.

In [17]:
test = datamart.loc[datamart.first_view_ts.notna()].copy()
test["first_view_ts"] = pd.to_datetime(test["first_view_ts"]).astype("datetime64[ns]")
test.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 6 to 120
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     str           
 1   labname          59 non-null     str           
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 2.3 KB


In [18]:
control = datamart.loc[datamart["first_view_ts"].isna()].copy()
mean_view_ts = test["first_view_ts"].mean()

mask = control["uid"].notna()
control.loc[mask, "first_view_ts"] = control.loc[mask, "first_view_ts"].fillna(
    mean_view_ts
)

control["first_view_ts"] = control["first_view_ts"].astype("datetime64[ns]")
test.to_sql("test", base_connect, if_exists="replace", index=False)
control.to_sql("control", base_connect, if_exists="replace", index=False)

control.info()


<class 'pandas.DataFrame'>
Index: 87 entries, 0 to 145
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     str           
 1   labname          81 non-null     str           
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), str(2)
memory usage: 3.4 KB


In [19]:
base_connect.close()